# INFOSYS 722 Iteration 4 – BDAS
Diabetes prediction on **AWS EC2 + Apache Spark + PySpark + Spark MLlib**.
Pipeline mirrors `src/`: data → preparation → transformation → modelling → evaluation.
Run from the project root (`INFOSYS722-BDAS-Diabetes/`).


In [ ]:
import sys
sys.path.insert(0, "src")
from spark_session import get_spark
from pyspark.sql.functions import col
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
spark = get_spark()
print("Spark Version:", spark.version)


## Step 2 – Data Understanding
Load D1a as a distributed Spark DataFrame and verify records, schema and statistics.


In [ ]:
df = (spark.read.option("header", True).option("inferSchema", True)
      .csv("data/brfss2023_diabetes_analysis.csv"))
print("Records:", df.count())
df.printSchema()
df.describe().show()


## Step 3 – Data Preparation
Drop target-missing records, resolve remaining blanks, persist cleaned Parquet.


In [ ]:
target = "Diabetes_binary"
print("Before cleaning:", df.count())
df_clean = df.dropna(subset=[target])
df_clean = df_clean.fillna(0)
print("After cleaning:", df_clean.count())
df_clean.write.mode("overwrite").parquet("data/clean_diabetes.parquet")
print("Saved: clean_diabetes.parquet")


## Step 4 – Data Transformation
Assemble the 19 cleaned predictors into a Spark MLlib `features` vector.


In [ ]:
df = spark.read.parquet("data/clean_diabetes.parquet")
features = [c for c in df.columns if c != target and c != "ID"]
print("Number of features:", len(features))
assembler = VectorAssembler(inputCols=features, outputCol="features")
df_model = assembler.transform(df).select("features", target)
df_model.show(5)
df_model.write.mode("overwrite").parquet("data/model_ready.parquet")
print("Saved: model_ready.parquet")


## Steps 5–6 – Method & Algorithm Selection
**Binary classification** with Spark MLlib: Logistic Regression (linear scorer) and Random Forest, 100 trees (distributed benchmark).


In [ ]:
df_model = spark.read.parquet("data/model_ready.parquet")
df_model = df_model.withColumnRenamed(target, "label")
train, test = df_model.randomSplit([0.8, 0.2], seed=42)
print("Training:", train.count())
print("Testing:", test.count())


In [ ]:
lr = LogisticRegression(featuresCol="features", labelCol="label")
lr_model = lr.fit(train)
lr_prediction = lr_model.transform(test)
lr_prediction.select("label", "prediction", "probability").show(10)
lr_prediction.write.mode("overwrite").parquet("output/lr_prediction")


In [ ]:
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, seed=42)
rf_model = rf.fit(train)
rf_prediction = rf_model.transform(test)
rf_prediction.select("label", "prediction", "probability").show(10)
rf_prediction.write.mode("overwrite").parquet("output/rf_prediction")
print("Model output saved")


## Step 8 – Interpretation & Evaluation
Confusion matrix, AUC and accuracy for the Random Forest on the held-out partition.


In [ ]:
prediction = spark.read.parquet("output/rf_prediction")
prediction.groupBy("label", "prediction").count().show()
auc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC").evaluate(prediction)
accuracy = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy").evaluate(prediction)
print("AUC:", auc)
print("Accuracy:", accuracy)
import pandas as pd
pd.DataFrame({"model": ["Random Forest"], "AUC": [auc], "Accuracy": [accuracy]}).to_csv("output/bdas_metrics.csv", index=False)
print("Saved: bdas_metrics.csv")


In [ ]:
spark.stop()
